# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates data loading and exploratory analysis using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library on the FAIR² Mental Health Survey Dataset collected in Kilifi County, Kenya.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata
meta = dataset.metadata
print(f"Dataset Name: {meta.name}")
print(f"Dataset Description: {meta.description}\n")
print(f"Published Date: {getattr(meta, 'datePublished', None)}")
print(f"Version: {getattr(meta, 'version', None)}")
print(f"License: {getattr(meta, 'license', None)}")


## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s by browsing the Croissant schema.

In [ ]:
# List available record sets by id and name
record_sets = dataset.record_sets()
record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name')}")
    record_set_ids.append(rs['@id'])
    # List fields (columns) for each RecordSet
    print("  Fields (columns):")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            field_id = field.get('@id')
            field_name = field.get('name')
        else:
            field_id = field
            field_name = None
        print(f"    - Field @id: {field_id}, name: {field_name}")
    print()


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references use `@id` for record sets and fields.

In [ ]:
# Collect all record set @ids from previous cell
# (If there is only one main record set, we use that)

dataframes = {}
for record_set_id in record_set_ids:
    # Pull records for this set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet @id: {record_set_id} with {len(df)} rows and columns:")
        print(df.columns.tolist())
        # Display the first few rows
        display(df.head())


## 4. Exploratory Data Analysis (EDA)
This section demonstrates common analysis operations such as filtering, normalization, and grouping using the actual field `@id`s. Change the field `@id` variables as needed based on what is shown in the overview step.

In [ ]:
# Example: Analyze GAD-7 scores
# Update the following variable to match the GAD-7 numerical score field's @id as found in dataframes.keys() and columns
main_record_set_id = record_set_ids[0]  # If multiple, choose the relevant one

df = dataframes[main_record_set_id]

# Search for numeric fields (e.g. GAD-7 score, PHQ-9 score) in the columns
print("Data columns available:", df.columns.tolist())

# Let's suppose one column has '@id' like 'gad7_score' (change as per available column @id after review)
numeric_field = None
for col in df.columns:
    if 'gad' in col.lower() and 'score' in col.lower():
        numeric_field = col
        break
if not numeric_field:
    # pick a numeric field
    for col in df.select_dtypes(include='number').columns:
        numeric_field = col
        break
print(f"Using field for EDA: {numeric_field}\n")

# Example filter: scores > 10
if numeric_field:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean())
        / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by e.g., gender or other demographic field using @id (make sure the field exists)
    group_field = None
    # Try to find a field containing 'gender' or similar
    for col in df.columns:
        if 'gender' in col.lower():
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().rename('mean_score')
        print(f"Mean {numeric_field} by {group_field}:")
        display(grouped_df)


## 5. Visualization
Visualize distributions or relationships from the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the main numeric field (e.g., GAD-7 score)
if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field found, plot boxplot
    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
We have loaded, inspected, and performed basic exploratory analysis on the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya. The dataset includes key survey-based mental health indicators (e.g., GAD-7, PHQ-9, PSQ scores) with demographic detail, enabling analyses such as symptom burden by group. The flexible use of `mlcroissant` ensures that all fields and entities are referenced by their `@id` as recommended, for reproducibility and schema alignment.